In [1]:
import numpy as np

# Paramètres du câble
Nx = 100          # nombre de compartiments
dx = 0.01         # cm
dt = 0.01         # ms
Tmax = 10         # ms

Cm = 1.0          # µF/cm²
D = 0.1           # diffusion intra-cellulaire (cm²/ms)

# Potentiel membranaire initial
V = np.zeros(Nx)

# --- Extracellular layer ---
class ExtracellularLayer:
    def __init__(self, Nx, xg=0.0001, xc=0.0, xraxial=1.0, e=None):
        self.vext = np.zeros(Nx)
        self.xg = xg          # mho/cm²
        self.xc = xc          # µF/cm²
        self.xraxial = xraxial # MΩ/cm
        self.e = np.zeros(Nx) if e is None else e
        # Calcul de D_ext pour diffusion 1D
        Ceff = 2 * np.pi * 0.005 * xc if xc > 0 else 1.0  # µF/cm (rayon 50 µm)
        Raxial = xraxial * 1e6  # Ω/cm
        self.Dext = 1.0 / (Raxial * Ceff) if xc > 0 else 0.0

layer = ExtracellularLayer(Nx)

# Courant ionique simple (passif)
gL = 0.1  # mho/cm²
EL = -65  # mV
def I_ion(V):
    return gL * (V - EL)

# Laplacien 1D
def laplacian(u):
    lap = np.zeros_like(u)
    lap[1:-1] = (u[2:] - 2*u[1:-1] + u[:-2]) / dx**2
    return lap

# Simulation
time = np.arange(0, Tmax, dt)
for t in time:
    # Extracellular courant pour le membrane
    I_ext_mem = layer.xg * ((V + layer.vext) - layer.e)
    # Ajouter capacitive term si xc>0
    if layer.xc > 0:
        dVdt = (V - V) / dt  # backward difference placeholder
        I_ext_mem += layer.xc * dVdt

    # Mise à jour du potentiel membranaire
    dV = (D * laplacian(V) - I_ion(V) - I_ext_mem) / Cm
    V += dt * dV

    # Mise à jour du potentiel extracellulaire (xc=0 ici pour simplifier)
    layer.vext += dt * (-layer.xg / 1.0 * (V + layer.vext - layer.e) + layer.Dext * laplacian(layer.vext))

# Exemple de sortie
print("V final:", V)
print("V extracellulaire final:", layer.vext)


V final: [-41.08263367 -41.08263367 -41.08263367 -41.08263367 -41.08263367
 -41.08263367 -41.08263367 -41.08263367 -41.08263367 -41.08263367
 -41.08263367 -41.08263367 -41.08263367 -41.08263367 -41.08263367
 -41.08263367 -41.08263367 -41.08263367 -41.08263367 -41.08263367
 -41.08263367 -41.08263367 -41.08263367 -41.08263367 -41.08263367
 -41.08263367 -41.08263367 -41.08263367 -41.08263367 -41.08263367
 -41.08263367 -41.08263367 -41.08263367 -41.08263367 -41.08263367
 -41.08263367 -41.08263367 -41.08263367 -41.08263367 -41.08263367
 -41.08263367 -41.08263367 -41.08263367 -41.08263367 -41.08263367
 -41.08263367 -41.08263367 -41.08263367 -41.08263367 -41.08263367
 -41.08263367 -41.08263367 -41.08263367 -41.08263367 -41.08263367
 -41.08263367 -41.08263367 -41.08263367 -41.08263367 -41.08263367
 -41.08263367 -41.08263367 -41.08263367 -41.08263367 -41.08263367
 -41.08263367 -41.08263367 -41.08263367 -41.08263367 -41.08263367
 -41.08263367 -41.08263367 -41.08263367 -41.08263367 -41.08263367
 